In [0]:

# Project         : Procurement Analytics using Databricks & Power BI
# Layer           : Bronze
# Notebook        : bronze_d
# Source          : products.csv
# Target          : procurement.bronze.bronze_products
# Audit Table     : procurement.audit.duplicate_products
#
# Author          : V R Mutyala
# Created Date    : 21-Jul-2026
# Last Modified   : 21-Jul-2026
#
# Description
# -----------
# This notebook loads raw department master data into the Bronze layer.
# It validates the source data, separates duplicate records, adds audit columns, and stores the results as Delta tables.

# ==============================================================================
# Business Objective
# ==============================================================================
#
# Load Products master data from the source CSV file into the Bronze layer.
#
# Preserve the raw source data with minimal transformations.
#
# Detect duplicate Products IDs and store them in the Audit schema for business review.
#
# Add audit columns to support data lineage and traceability.
#
# Create a reliable Bronze Delta table that will serve as the source for the Silver layer.
#
# ==============================================================================

In [0]:
%run ../01_Config/Config

In [0]:
%run ../05_Helper_Functions/Helper_functions

In [0]:
print(CATALOG)
print(BRONZE_PRODUCTS)
print(AUDIT_DUPLICATE_PRODUCTS)
print(PRODUCTS_FILE)

In [0]:
# Import Libraries and Widgets
from pyspark.sql import DataFrame
from pyspark.sql.functions import (col, lit,current_timestamp,when,count,trim)
from pyspark.sql.types import (StructType, StructField, StringType,IntegerType,DoubleType,DecimalType)
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

In [0]:
#Products schema
products_schema = StructType([
    StructField("product_id", StringType(), False),
    StructField("product_name", StringType(), True),
    StructField("category", StringType(), True),
    StructField("unit_of_measure", StringType(), True),
    StructField("standard_unit_price", DecimalType(18,2), True),
    StructField("currency", StringType(), True)
])

# Read Products master data from landing volume
bronze_products_df = (spark.read
    .format("csv")
    .option("header", True)
    .schema(products_schema)
    .load(PRODUCTS_FILE)
)

#Source Data validation 
print(f"Total Records : {bronze_products_df.count()}")

print("\nSchema:")
bronze_products_df.printSchema()

print("\nColumns:")
print(bronze_products_df.columns)

print("\nSampledata:")
display(bronze_products_df.limit(10))

In [0]:
# Check the NULL and Blank Product_ids
null_blank_product_id = bronze_products_df.filter(col("product_id").isNull() | (trim(col("product_id")) == ""))

print(f"Total NULL or Blank product_ids : {null_blank_product_id.count()}")

display(null_blank_product_id)

In [0]:
#Check the NULL and Blank product name
null_blank_product_name = bronze_products_df.filter(col("product_name").isNull() | (trim(col("product_name")) == ""))

print(f"Total NULL or Blank product_name : {null_blank_product_name.count()}")

display(null_blank_product_name)

In [0]:
#Check the negative standard_unit_price

negative_standard_unit_price = bronze_products_df.filter(col("standard_unit_price") < 0)

print(f"Total negative standard_unit_price : {negative_standard_unit_price .count()}")

display(negative_standard_unit_price )


In [0]:
# ============================================================
# Identify Duplicate Products IDs
# ============================================================

duplicate_product_keys = (
    bronze_products_df
        .groupBy("product_id")
        .count()
        .filter(col("count") > 1)
        .withColumnRenamed("count", "duplicate_count")
)

display(duplicate_product_keys)

In [0]:
# ============================================================
# Identify Duplicate Products Records
# Business Rule: Keep the first occurrence of each Department ID and identify subsequent records as  duplicates.
# ============================================================

window_spec = Window.partitionBy("product_id").orderBy("product_id")

products_rank_df = (
    bronze_products_df
        .withColumn(
            "row_num",
            row_number().over(window_spec)
        )
)
display(products_rank_df)

In [0]:
# ============================================================
# Retrieve Duplicate Product Records
# ============================================================

duplicate_products = (
    products_rank_df
        .filter(col("row_num") > 1)
        .drop("row_num")
)

print(f"Duplicate Product Records : {duplicate_products.count()}")

display(duplicate_products)

In [0]:
# ============================================================
# Add Audit Metadata
# ============================================================
duplicate_products = (
    duplicate_products
        .withColumn("audit_timestamp", current_timestamp())
        .withColumn("source_table", lit("Products"))
        .withColumn("pipeline_layer", lit("Bronze"))
        .withColumn("issue_type", lit("Duplicate Record"))
)

display(duplicate_products)

In [0]:
# ============================================================
# Write Duplicate Records to Audit Table
# ============================================================

duplicate_count = duplicate_products.count()

if duplicate_count > 0:

    write_delta(
        df = duplicate_products,
         table_name = AUDIT_DUPLICATE_PRODUCTS
    )

    print(f"Successfully written {duplicate_count} duplicate record(s) to {AUDIT_DUPLICATE_PRODUCTS}")

else:

    print("No duplicate products records found. Audit table not created.")

In [0]:
# ============================================================
# Add Bronze Audit Columns
# ============================================================

bronze_products_final_df = (
    bronze_products_df
        .withColumn("load_timestamp", current_timestamp())
        .withColumn("source_file", lit("Products.csv"))
)

In [0]:
# ============================================================
# Write Bronze Delta Table
# ============================================================

write_delta(df=bronze_products_final_df,table_name=BRONZE_PRODUCTS)

In [0]:
# ============================================================
# Validate Bronze Delta Table
# ============================================================

bronze_products = spark.table(BRONZE_PRODUCTS)

print(f"Total Bronze Records : {bronze_products.count()}")

display(bronze_products)

In [0]:
# ============================================================
# Bronze Product complete summary
# ============================================================
print("=" * 60)
print("Bronze Productt Load Completed Successfully")
print("=" * 60)

print(f"Landing Records : {bronze_products_df.count()}")

print(f"Audit Records : {duplicate_products.count()}")

print(f"Bronze Records : {spark.table(BRONZE_PRODUCTS).count()}")